# FraudMesh on Kaggle

Graph-based fraud ring detection (XGBoost baseline -> XGBoost + graph
features -> GraphSAGE) run against the real IEEE-CIS Fraud Detection
competition data.

**Before running this notebook:**
1. Add the competition data: *Add Input -> Competitions -> IEEE-CIS Fraud
   Detection* (or the equivalent Kaggle Dataset mirror of
   `train_transaction.csv` / `train_identity.csv`).
2. Get this repo's code onto the notebook, either:
   - **Recommended:** zip this project folder and upload it as a private
     Kaggle Dataset via *Add Input -> Datasets -> New Dataset*, then attach
     it here, or
   - Push it to a Git remote and let the clone cell below pull it into
     `/kaggle/working`.
3. Turn on a GPU accelerator (*Settings -> Accelerator -> GPU*) — GraphSAGE
   will use it automatically if present, and will fall back to CPU
   otherwise.

Everything below is read-only against `/kaggle/input`; all outputs
(`results/*.json`, `models/graphsage.pt`) are written under
`/kaggle/working`, handled automatically by `config.py`'s Kaggle-path
detection.


In [ ]:
import os
import sys
import glob
import subprocess

def find_project_root():
    """Locate the FraudMesh checkout: either uploaded as a Kaggle Dataset
    (mounted read-only under /kaggle/input/<dataset-name>/) or already
    cloned into /kaggle/working."""
    patterns = [
        "/kaggle/input/*/config.py",
        "/kaggle/input/*/*/config.py",
        "/kaggle/working/*/config.py",
        "/kaggle/working/*/*/config.py",
    ]
    for pattern in patterns:
        for path in glob.glob(pattern):
            root = os.path.dirname(path)
            if os.path.isdir(os.path.join(root, "src")):
                return root
    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    # Fallback: clone from GitHub if the repo wasn't attached as a Kaggle
    # Dataset. Only picks up whatever's actually pushed to this URL — if
    # you're relying on this path, push your local changes first.
    REPO_URL = "https://github.com/dhyanagni2001-commits/FraudMesh.git"
    print(f"No local checkout found under /kaggle/input or /kaggle/working; "
          f"cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/FraudMesh"], check=True)
    PROJECT_ROOT = "/kaggle/working/FraudMesh"

print("Using project root:", PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Kaggle preinstalls pandas/numpy/xgboost/torch/networkx already matched to
# its image — only torch_geometric is actually missing. See
# requirements-kaggle.txt for why we don't `pip install -r requirements.txt`
# here.
!pip install -q -r requirements-kaggle.txt


In [ ]:
import config
print("ON_KAGGLE:", config.ON_KAGGLE)
print("DATA_DIR:", config.DATA_DIR)
print("RESULTS_DIR:", config.RESULTS_DIR)
print("MODELS_DIR:", config.MODELS_DIR)
assert os.path.exists(config.TRANSACTION_CSV), (
    f"Couldn't find train_transaction.csv under {config.DATA_DIR}. "
    "Attach the IEEE-CIS Fraud Detection data via Add Input first."
)


## Run the full pipeline

Runs all four phases against the real data mounted above: XGBoost baseline
-> graph-augmented XGBoost -> GraphSAGE -> fraud-ring case study. This is
the same `scripts/run_pipeline.sh` used locally; add `--sample-frac 0.2` (or
similar) to any of the phases below first if you want a fast smoke-test
pass on a subsample before committing to the full ~590k-row run.


In [ ]:
!bash scripts/run_pipeline.sh


## Ablation summary

In [ ]:
import json
import pandas as pd

rows = []
for label, fname in [
    ("XGBoost (baseline, no graph)", "baseline_metrics.json"),
    ("XGBoost + graph features", "graph_features_metrics.json"),
    ("GraphSAGE (end-to-end)", "graphsage_metrics.json"),
]:
    with open(os.path.join(config.RESULTS_DIR, fname)) as f:
        d = json.load(f)
    rows.append({"model": label, **d["metrics"]})

pd.DataFrame(rows).set_index("model")


## Fraud rings found (case study)

In [ ]:
with open(os.path.join(config.RESULTS_DIR, "case_study_rings.json")) as f:
    rings = json.load(f)

for i, r in enumerate(rings, 1):
    print(f"Ring #{i}")
    print(f"  transactions:   {r['n_transactions']}")
    print(f"  fraud rate:     {r['fraud_rate']:.1%}" if r["fraud_rate"] is not None else "  fraud rate: n/a")
    print(f"  avg amount:     ${r['avg_amount']:.2f}")
    print(f"  total amount:   ${r['total_amount']:.2f}")
    print(f"  time span:      {r['time_span_hours']:.1f} hours")
    print(f"  unique cards:   {r['unique_cards']}")
    print(f"  unique devices: {r['unique_devices']}")
    print(f"  linked via:     {', '.join(r['shared_entity_types'])}")
    print()


## Notes

- `models/graphsage.pt` and every `results/*.json` are now under
  `/kaggle/working` — use *Save Version* (or the Output tab) to persist
  them past the session.
- `src/serve.py` (the FastAPI scoring layer) isn't meant to run inside a
  notebook cell — it's a long-lived server process. Download
  `models/graphsage.pt` from the Output tab and run it locally per the
  README's Serving section if you want to try it.
- Re-run a single phase instead of the whole pipeline with, e.g.,
  `!python3 src/train_graphsage.py --sample-frac 0.3` for faster iteration.
